In [1]:
import qlib
import pandas as pd

from qlib.data.dataset.handler import DataHandlerLP

qlib.init(provider_uri="~/.qlib/qlib_data/cn_data")

[8022:MainThread](2026-01-29 16:26:55,663) INFO - qlib.Initialization - [config.py:452] - default_conf: client.
[8022:MainThread](2026-01-29 16:26:56,101) INFO - qlib.Initialization - [__init__.py:82] - qlib successfully initialized based on client settings.
[8022:MainThread](2026-01-29 16:26:56,102) INFO - qlib.Initialization - [__init__.py:84] - data_path={'__DEFAULT_FREQ': PosixPath('/home/chainsmoker/.qlib/qlib_data/cn_data')}


In [48]:
from qlib.data import D

instruments = D.instruments(market='all')
start_time="2025-01-01"
end_time="2025-12-31"
fit_start_time="2025-01-01"
fit_end_time="2025-8-31"
test_start_time = "2025-09-01"
test_end_time = "2025-10-31"

stock_list = D.list_instruments(instruments=instruments,
                                start_time=start_time,
                                end_time=end_time,
                                as_list=True)
print(stock_list[-5:])

['SZ301667', 'SZ301668', 'SZ301678', 'SZ301687', 'SZ302132']


In [49]:
from qlib.data.dataset.loader import QlibDataLoader

data_loader = QlibDataLoader(
    config={
        "feature": [
            "$open",
            "$high",
            "$low",
            "$close",
            "$volume",
        ],
        "label": [
            "Ref($close, -2) / Ref($close, -1) - 1",  # 未来1日收益（示例）
        ],
    }
)


In [50]:
handler = DataHandlerLP(
    instruments=instruments,
    start_time=start_time,
    end_time=end_time,
    data_loader=data_loader,
)

[8022:MainThread](2026-01-29 20:43:32,948) INFO - qlib.timer - [log.py:127] - Time cost: 3.239s | Loading data Done
[8022:MainThread](2026-01-29 20:43:32,949) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done
[8022:MainThread](2026-01-29 20:43:32,950) INFO - qlib.timer - [log.py:127] - Time cost: 3.241s | Init data Done


In [51]:
# 取出“日级特征表”和“日级标签表”
df_x = handler.fetch(col_set="feature")
df_y = handler.fetch(col_set="label")


print("=== features（日级 MultiIndex 表）===")
print(df_x.head())
print("index:", df_x.index.names, "shape:", df_x.shape, "columns:", list(df_x.columns))

print("\n=== label（日级 MultiIndex 表）===")
print(df_y.head())
print("index:", df_y.index.names, "shape:", df_y.shape, "columns:", list(df_y.columns))

=== features（日级 MultiIndex 表）===
                           $open      $high       $low     $close    $volume
datetime   instrument                                                       
2025-01-02 BJ920000    18.299999  18.500000  17.379999  17.900000   993934.0
           BJ920001    16.350000  17.440001  16.000000  16.600000  3958586.0
           BJ920002    67.500000  69.989998  66.989998  67.980003   791137.0
           BJ920006    24.000000  24.980000  23.510000  24.820000  5070593.0
           BJ920008    30.280001  30.660000  28.969999  29.280001  1103154.0
index: ['datetime', 'instrument'] shape: (1273246, 5) columns: ['$open', '$high', '$low', '$close', '$volume']

=== label（日级 MultiIndex 表）===
                       Ref($close, -2) / Ref($close, -1) - 1
datetime   instrument                                       
2025-01-02 BJ920000                                -0.016234
           BJ920001                                -0.128030
           BJ920002                       

In [52]:
import numpy as np
from qlib.data.dataset import TSDatasetH

L = 60
dataset = TSDatasetH(
    handler=handler,
    segments={
        "train": (fit_start_time, fit_end_time),
        "test":  (test_start_time, test_end_time),
    },
    step_len=L,
)

# 准备 train 段样本（返回一个可索引对象）
train_set = dataset.prepare("train")
test_set = dataset.prepare("test")

print("train_set length:", len(train_set))
print("test_set length:", len(test_set))


train_set length: 841361
test_set length: 204956


In [53]:
# 随机取一个时间窗口样本
sample = train_set[100]          # (20, 6)

x = sample[:, :5].astype(np.float32)   # (20, 5) OHLCV窗口
y = np.float32(sample[:, -1])          # 窗口末端对应的label（标量）

print("x.shape:", x.shape)     
print("y.shape:", y.shape)

print("x:", x)     
print("y:", y)


x.shape: (60, 5)
y.shape: (60,)
x: [[2.3969999e+01 2.4900000e+01 2.3600000e+01 2.4830000e+01 3.8111380e+06]
 [2.4450001e+01 2.5070000e+01 2.3879999e+01 2.4930000e+01 3.4468350e+06]
 [2.5139999e+01 2.8799999e+01 2.5059999e+01 2.6820000e+01 9.6977270e+06]
 [2.6400000e+01 3.4349998e+01 2.5879999e+01 3.2869999e+01 1.7302274e+07]
 [3.1110001e+01 3.1549999e+01 2.8690001e+01 2.9629999e+01 1.2372829e+07]
 [2.8510000e+01 2.9879999e+01 2.8150000e+01 2.8250000e+01 7.6067590e+06]
 [2.8799999e+01 2.8930000e+01 2.7500000e+01 2.7910000e+01 4.7379430e+06]
 [2.7600000e+01 2.9090000e+01 2.7010000e+01 2.7180000e+01 6.1836400e+06]
 [2.6580000e+01 2.7000000e+01 2.5520000e+01 2.5600000e+01 5.7975800e+06]
 [2.5940001e+01 2.5950001e+01 2.3750000e+01 2.3820000e+01 4.2641810e+06]
 [2.3889999e+01 2.5799999e+01 2.3639999e+01 2.5520000e+01 3.8651530e+06]
 [2.5320000e+01 2.6070000e+01 2.4900000e+01 2.5870001e+01 3.5142930e+06]
 [2.5799999e+01 2.6160000e+01 2.4830000e+01 2.4900000e+01 3.1387570e+06]
 [2.4709999e+01 

In [59]:
import numpy as np

def nan_ratio(qlib_ts_set, n=500):
    total = 0
    nan_cnt = 0
    for i in range(len(qlib_ts_set)):
        s = qlib_ts_set[i]
        x = s[:, :5]
        nan_cnt += np.isnan(x).sum()
        total += x.size
    return nan_cnt / total

print("train NaN ratio:", nan_ratio(train_set, len(train_set)))


train NaN ratio: 0.1858339048280108


In [54]:
import torch
from torch.utils.data import Dataset, DataLoader

class QlibOHLCVDataset(Dataset):
    """
    输入: qlib_ts_set[i] -> numpy (L, 6) = [O,H,L,C,V,label]
    输出：
      x: torch.FloatTensor (L, 5)
      y: torch.FloatTensor ()  (标量)
    """
    def __init__(self, qlib_ts_set, valid_idx, ohlcv_dim=5, label_col=5):
        self.ds = qlib_ts_set
        self.valid_idx = valid_idx # 只提取不为Nan的数据
        self.ohlcv_dim = ohlcv_dim
        self.label_col = label_col

    def __len__(self):
        return len(self.valid_idx)

    def __getitem__(self, j):
        i = int(self.valid_idx[j])
        s = self.ds[i]  # numpy (L,6)
        x = torch.from_numpy(s[:, :self.ohlcv_dim].astype(np.float32))   # (L,5)
        y = torch.tensor(np.float32(s[-1, self.label_col]), dtype=torch.float32)  # scalar
        return x, y


In [56]:
def build_valid_indices(qlib_ts_set, ohlcv_dim=5, label_col=5):
    valid = []
    for i in range(len(qlib_ts_set)):
        s = qlib_ts_set[i]  # (L,6)
        x = s[:, :ohlcv_dim]
        y = s[-1, label_col]
        if np.isfinite(x).all() and np.isfinite(y):
            valid.append(i)
    return np.array(valid, dtype=np.int64)

In [60]:
train_set = dataset.prepare("train")
test_set  = dataset.prepare("test")

valid_train = build_valid_indices(train_set)
valid_test  = build_valid_indices(test_set)

print("train kept:", len(valid_train), " / ", len(train_set), " ratio:", len(valid_train)/len(train_set))
print("test kept:",  len(valid_test),  " / ", len(test_set),  " ratio:", len(valid_test)/len(test_set))

torch_train = QlibOHLCVDataset(train_set, valid_train)
torch_test  = QlibOHLCVDataset(test_set,  valid_test)


train kept: 520884  /  841361  ratio: 0.6190969155927123
test kept: 201104  /  204956  ratio: 0.9812057222037901


In [61]:
train_loader = DataLoader(torch_train, batch_size=64, shuffle=True, drop_last=True)
test_loader  = DataLoader(torch_test,  batch_size=64, shuffle=False)

In [62]:
import torch
import torch.nn as nn

class TorchFactorExtractor(nn.Module):
    """
    输入:  x_raw (B, L, 5) -> [open, high, low, close, volume]
    输出:  feats (B, L', F)
    """
    def __init__(self, wins=(5, 10, 20), eps=1e-6):
        super().__init__()
        self.wins = tuple(sorted(wins))
        self.eps = eps

    def _roll_mean_std(self, x, win):
        # x: (B, L)
        xu = x.unfold(dimension=1, size=win, step=1)     # (B, L-win+1, win)
        mean = xu.mean(dim=-1)                           # (B, L-win+1)
        std = xu.std(dim=-1, unbiased=False)             # (B, L-win+1)
        return mean, std

    def forward(self, x):
        o = x[..., 0]
        h = x[..., 1]
        l = x[..., 2]
        c = x[..., 3]
        v = x[..., 4]

        # 基础逐日特征
        amp = (h - l) / (c + self.eps)                   # (B, L)
        vlog = torch.log1p(torch.clamp(v, min=0.0))      # (B, L)
        logret = torch.log((c[:, 1:] + self.eps) / (c[:, :-1] + self.eps))  # (B, L-1)

        # rolling 特征
        roll_feats = []
        for w in self.wins:
            ma, sd = self._roll_mean_std(c, w)           # (B, L-w+1)
            z = (c[:, w-1:] - ma) / (sd + self.eps)      # 对齐到 (B, L-w+1)
            roll_feats.append((ma, sd, z))

        # 对齐长度到最短 L'
        L = c.shape[1]
        Lp = min(
            L,                 # amp/vlog
            L - 1,             # logret
            *[L - w + 1 for w in self.wins]  # rolling
        )

        amp = amp[:, -Lp:]
        vlog = vlog[:, -Lp:]
        logret = logret[:, -Lp:]

        feat_list = [logret, amp, vlog]  # 3 个

        for (ma, sd, z), w in zip(roll_feats, self.wins):
            ma = ma[:, -Lp:]
            sd = sd[:, -Lp:]
            z  = z[:,  -Lp:]
            feat_list.extend([ma, sd, z])  # 每个窗口 3 个

        feats = torch.stack(feat_list, dim=-1)  # (B, Lp, F)
        return feats


In [63]:
class LSTMPredictor(nn.Module):
    def __init__(self, feat_dim, hidden=64, dropout=0.0):
        super().__init__()
        self.lstm = nn.LSTM(input_size=feat_dim, hidden_size=hidden, batch_first=True)
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden, 1)

    def forward(self, feats):
        out, _ = self.lstm(feats)         # (B, Lp, hidden)
        last = out[:, -1, :]              # (B, hidden)
        last = self.drop(last)
        yhat = self.fc(last).squeeze(-1)  # (B,)
        return yhat


In [64]:
class End2EndModel(nn.Module):
    def __init__(self, wins=(5,10,20), hidden=64):
        super().__init__()
        self.extractor = TorchFactorExtractor(wins=wins)
        feat_dim = 3 + 3*len(wins)
        self.predictor = LSTMPredictor(feat_dim=feat_dim, hidden=hidden)

    def forward(self, x_raw):
        feats = self.extractor(x_raw)
        return self.predictor(feats)


In [66]:
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"
model = End2EndModel(wins=(5,10,20), hidden=64).to(device)

xb, yb = next(iter(train_loader))        # xb: (B, 60, 5), yb: (B,)
xb = xb.to(device).detach().requires_grad_(True)
yb = yb.to(device)

yhat = model(xb)
loss = F.mse_loss(yhat, yb)
loss.backward()

print("xb:", xb.shape, "yb:", yb.shape)
print("loss:", float(loss))
print("xb.grad is None?", xb.grad is None)
print("grad mean abs:", float(xb.grad.abs().mean()))
print("grad max abs:", float(xb.grad.abs().max()))
print("grad has nan?", bool(torch.isnan(xb.grad).any()))


xb: torch.Size([64, 60, 5]) yb: torch.Size([64])
loss: 0.011261341162025928
xb.grad is None? False
grad mean abs: 1.730504777697206e-06
grad max abs: 0.004470984451472759
grad has nan? False


In [67]:
eps = 1e-3  # 先用很小的扰动
xb_adv = xb.detach() + eps * xb.grad.sign()

with torch.no_grad():
    yhat_clean = model(xb.detach())
    yhat_adv = model(xb_adv)
    loss_clean = F.mse_loss(yhat_clean, yb)
    loss_adv = F.mse_loss(yhat_adv, yb)

print("loss clean:", float(loss_clean))
print("loss adv  :", float(loss_adv))


loss clean: 0.011261341162025928
loss adv  : 0.011294384486973286


In [68]:
from qlib.contrib.data.handler import Alpha158

data_config = {
    "start_time": start_time,
    "end_time": end_time,
    "fit_start_time": fit_start_time,
    "fit_end_time": fit_end_time,
    "instruments": instruments,
}

h = Alpha158(**data_config)
print(h.get_cols())

[8022:MainThread](2026-01-30 16:53:10,473) INFO - qlib.timer - [log.py:127] - Time cost: 40.302s | Loading data Done
[8022:MainThread](2026-01-30 16:53:11,626) INFO - qlib.timer - [log.py:127] - Time cost: 0.276s | DropnaLabel Done
[8022:MainThread](2026-01-30 16:53:11,916) INFO - qlib.timer - [log.py:127] - Time cost: 0.289s | CSZScoreNorm Done
[8022:MainThread](2026-01-30 16:53:11,918) INFO - qlib.timer - [log.py:127] - Time cost: 1.443s | fit & process data Done
[8022:MainThread](2026-01-30 16:53:11,919) INFO - qlib.timer - [log.py:127] - Time cost: 41.749s | Init data Done


['KMID', 'KLEN', 'KMID2', 'KUP', 'KUP2', 'KLOW', 'KLOW2', 'KSFT', 'KSFT2', 'OPEN0', 'HIGH0', 'LOW0', 'VWAP0', 'ROC5', 'ROC10', 'ROC20', 'ROC30', 'ROC60', 'MA5', 'MA10', 'MA20', 'MA30', 'MA60', 'STD5', 'STD10', 'STD20', 'STD30', 'STD60', 'BETA5', 'BETA10', 'BETA20', 'BETA30', 'BETA60', 'RSQR5', 'RSQR10', 'RSQR20', 'RSQR30', 'RSQR60', 'RESI5', 'RESI10', 'RESI20', 'RESI30', 'RESI60', 'MAX5', 'MAX10', 'MAX20', 'MAX30', 'MAX60', 'MIN5', 'MIN10', 'MIN20', 'MIN30', 'MIN60', 'QTLU5', 'QTLU10', 'QTLU20', 'QTLU30', 'QTLU60', 'QTLD5', 'QTLD10', 'QTLD20', 'QTLD30', 'QTLD60', 'RANK5', 'RANK10', 'RANK20', 'RANK30', 'RANK60', 'RSV5', 'RSV10', 'RSV20', 'RSV30', 'RSV60', 'IMAX5', 'IMAX10', 'IMAX20', 'IMAX30', 'IMAX60', 'IMIN5', 'IMIN10', 'IMIN20', 'IMIN30', 'IMIN60', 'IMXD5', 'IMXD10', 'IMXD20', 'IMXD30', 'IMXD60', 'CORR5', 'CORR10', 'CORR20', 'CORR30', 'CORR60', 'CORD5', 'CORD10', 'CORD20', 'CORD30', 'CORD60', 'CNTP5', 'CNTP10', 'CNTP20', 'CNTP30', 'CNTP60', 'CNTN5', 'CNTN10', 'CNTN20', 'CNTN30', 'CNT

In [69]:
print(h)

In [71]:
from qlib.contrib.data.loader import Alpha158DL

fields, names = Alpha158DL.get_feature_config()

df = pd.DataFrame({"name": names, "expression": fields})

df.to_csv("alpha158_name_expression.csv", index=False)
print("saved to alpha158_name_expression.csv")

saved to alpha158_name_expression.csv


# 离散算子的可微近似

# soft-max / soft-min

In [77]:
import torch

def smooth_max(x, delta=1):
    return torch.log(torch.exp(x * delta).sum()) / delta

x = torch.tensor([2.0, 4.0, 1.0], requires_grad=True)
y = smooth_max(x, 10)
print(y)
y.backward()
print(x.grad)

tensor(4., grad_fn=<DivBackward0>)
tensor([2.0612e-09, 1.0000e+00, 9.3576e-14])


# soft-rank

In [16]:
import torch
from fast_soft_sort.pytorch_ops import soft_rank, soft_sort

x = torch.tensor([[2., 4., 1., 3., 7., 5., 0.]], dtype=torch.float64, requires_grad=True)

for s in [2.0, 1.0, 0.5, 0.2, 0.1]:
    x.grad = None
    r = soft_rank(x, regularization_strength=s).squeeze(0)
    loss = (r**2).sum()
    loss.backward()
    print("s =", s, "soft_rank =", r.detach().numpy(), "grad =", x.grad.detach().numpy())


s = 2.0 soft_rank = [3.42857143 4.42857143 2.92857143 3.92857143 5.92857143 4.92857143
 2.42857143] grad = [[-0.57142857  0.42857143 -1.07142857 -0.07142857  1.92857143  0.92857143
  -1.57142857]]
s = 1.0 soft_rank = [3. 5. 2. 4. 7. 6. 1.] grad = [[-1.  3. -3.  1.  0.  5. -5.]]
s = 0.5 soft_rank = [3. 5. 2. 4. 7. 6. 1.] grad = [[0. 0. 0. 0. 0. 0. 0.]]
s = 0.2 soft_rank = [3. 5. 2. 4. 7. 6. 1.] grad = [[0. 0. 0. 0. 0. 0. 0.]]
s = 0.1 soft_rank = [3. 5. 2. 4. 7. 6. 1.] grad = [[0. 0. 0. 0. 0. 0. 0.]]


# soft-quantile

In [ ]:
import torch
import torch.nn.functional as F
from fast_soft_sort.pytorch_ops import soft_sort

def soft_quantile(values_2d, q: float,
                         reg_strength: float = 0.3,
                         pick_strength: float = 0.3,
                         eps: float = 1e-6):
    """
    values_2d: (B, n) float tensor, requires_grad ok
    q: [0,1]
    reg_strength: soft_sort 的正则强度（在标准化后可取 0.2~0.5）
    pick_strength: 位置选择的平滑度（越小越像硬取 pos）
    """
    x = values_2d
    mu = x.mean(dim=1, keepdim=True)
    std = x.std(dim=1, keepdim=True, unbiased=False)
    xz = (x - mu) / (std + eps)

    xs = soft_sort(xz, regularization_strength=reg_strength)  # (B, n)

    n = xs.size(1)
    pos = q * (n - 1)
    k = torch.arange(n, device=xs.device, dtype=xs.dtype)

    logits = -((k - pos) ** 2) / pick_strength
    w = F.softmax(logits, dim=-1)  # (n,)
    qz = (xs * w.view(1, -1)).sum(dim=1)  # (B,)

    # back to original scale
    return qz * (std.squeeze(1) + eps) + mu.squeeze(1)


In [ ]:
x = torch.tensor([[2.,4.,1.,3.,31.2,323.4,0.,9.,5.,4.]], dtype=torch.float64, requires_grad=True)
x.grad = None
q = soft_quantile(x, 0.5, reg_strength=0.3, pick_strength=0.3)
loss = (q - 2.5).pow(2).sum()
loss.backward()
print("q:", q.item())
print("grad std:", x.grad.std().item())
print("grad:", x.grad.detach().numpy())


q: 4.000000003087948
grad std: 0.6314511848176633
grad: [[ 3.08780068e-09  1.49809348e+00 -1.13943390e-16  1.90652439e-03
   1.95629340e-17  1.30493850e-15 -1.24728561e-16  3.08780071e-09
   1.90652439e-03  1.49809348e+00]]


# soft-greater / soft-less

In [29]:
import torch
import torch.nn.functional as F

def soft_greater(a: torch.Tensor, b: torch.Tensor, temperature: float = 0.1):
    """
    Differentiable approximation of I(a > b) using sigmoid((a-b)/T).
    a, b: broadcastable tensors
    returns: same broadcasted shape, values in (0,1)
    """
    T = temperature
    return torch.sigmoid((a - b) / T)

def soft_less(a: torch.Tensor, b: torch.Tensor, temperature: float = 0.1):
    """
    Differentiable approximation of I(a < b).
    """
    T = temperature
    return torch.sigmoid((b - a) / T)


In [31]:
def grad_check_soft_greater():
    a = torch.tensor([2.0, 4.0, 1.0, 3.0], requires_grad=True)
    b = torch.tensor([2.5, 2.5, 2.5, 2.5], requires_grad=True)

    g = soft_greater(a, b, temperature=0.1)   # (4,)
    loss = g.sum()
    loss.backward()

    print("g:", g.detach().cpu().numpy())
    print("a.grad:", a.grad.detach().cpu().numpy())
    print("b.grad:", b.grad.detach().cpu().numpy())
    print("nonzero a.grad:", int((a.grad.abs() > 1e-8).sum().item()))

grad_check_soft_greater()


g: [6.6928510e-03 9.9999964e-01 3.0590223e-07 9.9330717e-01]
a.grad: [6.6480562e-02 3.5762775e-06 3.0590215e-06 6.6480331e-02]
b.grad: [-6.6480562e-02 -3.5762775e-06 -3.0590215e-06 -6.6480331e-02]
nonzero a.grad: 4


# soft-IdxMax / soft-IdxMin

In [55]:
import torch
import torch.nn.functional as F

def hard_idxmax_st_onehot(x, tau=1.0, dim=-1):
    """
    Forward:
      idx_long = argmax(x)  (integer)
      y_st behaves like hard one-hot in forward
    Backward:
      gradients flow as if y_st were softmax(x/tau)
    Returns: (idx_long, y_st, y_soft)
    """
    y_soft = F.softmax(x / tau, dim=dim)  # differentiable
    idx_long = torch.argmax(x, dim=dim)   # integer

    y_hard = F.one_hot(idx_long, num_classes=x.size(dim)).to(dtype=x.dtype)
    # Make y_hard broadcastable if needed (one_hot puts class dim at end)
    if dim != -1 and dim != x.ndim - 1:
        # Move class dimension to 'dim'
        # one_hot output shape: (..., n_classes) with classes at last dim
        # we need it aligned with x
        perm = list(range(y_hard.ndim))
        perm.insert(dim, perm.pop(-1))
        y_hard = y_hard.permute(perm)
        y_soft = y_soft.permute(perm)

    y_st = (y_hard - y_soft).detach() + y_soft
    return idx_long, y_st, y_soft

def hard_idxmin_st_onehot(x, tau=1.0, dim=-1):
    # argmin(x) == argmax(-x)
    idx_long, y_st, y_soft = hard_idxmax_st_onehot(-x, tau=tau, dim=dim)
    return idx_long, y_st, y_soft

def st_select(x, y_st, dim=-1):
    """
    Select value(s) from x using y_st (hard in forward, soft in backward).
    Equivalent to gather along dim when y_st is one-hot along dim.
    """
    return (x * y_st).sum(dim=dim)

# ---------- Demo: forward hard, backward has gradients ----------
x = torch.tensor([2., 4., 1., 3.], requires_grad=True)

idx, y_st, y_soft = hard_idxmax_st_onehot(x, tau=0.5)
selected = st_select(x, y_st, dim=-1)   # forward == x[idx]

print("idx (hard int) =", idx.item())
print("selected (forward hard) =", selected.item(), "vs x[idx] =", x[idx].item())

selected.backward()
print("x.grad =", x.grad)


idx (hard int) = 1
selected (forward hard) = 4.0 vs x[idx] = 4.0
x.grad = tensor([-0.0585,  1.2684, -0.0122, -0.1978])
